In [8]:
import sys
from pathlib import Path
import json

if Path.cwd().name == 'chunking':
    PROJECT_ROOT = Path.cwd().parent.parent
else:
    PROJECT_ROOT = Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'parsers'))

In [17]:
with open('/Users/sarath/work/doc-parser/experiments/chunking/qa_pairs.json', 'r') as file:
    data = json.load(file)

In [20]:

for doc, q_list in data['documents'].items():
    

{'VAM-3852AO': {'pdf_path': 'docs/VAM-3852AO.pdf',
  'questions': [{'id': 'vam_01',
    'question': 'What is the ticker symbol for the S&P 500 Price Index?',
    'expected_keywords': ['SPX'],
    'answer_pages': [1],
    'type': 'factual'},
   {'id': 'vam_02',
    'question': 'What was the compound annual growth rate of the S&P 500 from 1993 to 2023?',
    'expected_keywords': ['8.05%', '8.05'],
    'answer_pages': [1],
    'type': 'chart_reading'},
   {'id': 'vam_03',
    'question': 'What was the annual return of the S&P 500 in 2022?',
    'expected_keywords': ['-19.44%', '-19.44'],
    'answer_pages': [2],
    'type': 'table_lookup'},
   {'id': 'vam_04',
    'question': 'What percentage of the S&P 500 is allocated to Information Technology?',
    'expected_keywords': ['28.90%', '28.90', '28.9%'],
    'answer_pages': [2],
    'type': 'chart_reading'},
   {'id': 'vam_05',
    'question': 'How many constituents does the S&P 500 index have?',
    'expected_keywords': ['503'],
    'answe

In [9]:
from chonkie import TableChunker, RecursiveChunker, RecursiveRules
import pymupdf4llm
from pymupdf_fast import parse_document as pymupdf_fast_parse

pdf_path = PROJECT_ROOT / 'docs' / 'hubspot-q4.pdf'

# Parse with both approaches
md_fast = pymupdf_fast_parse(pdf_path)
md_4llm = pymupdf4llm.to_markdown(str(pdf_path), show_progress=False)

print(f'pymupdf_fast:  {len(md_fast):>6} chars, {len(md_fast.splitlines()):>5} lines')
print(f'pymupdf4llm:   {len(md_4llm):>6} chars, {len(md_4llm.splitlines()):>5} lines')

Task was destroyed but it is pending!
task: <Task pending name='Task-130' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/sarath/work/doc-parser/venv_pdf/lib/python3.14/site-packages/ipykernel/utils.py:57> wait_for=<Task finished name='Task-131' coro=<Kernel.shell_main() done, defined at /Users/sarath/work/doc-parser/venv_pdf/lib/python3.14/site-packages/ipykernel/kernelbase.py:597> exception=KeyError('70e854ee-3d55-422c-98d9-479a5e5bc9e6')> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/sarath/work/doc-parser/venv_pdf/lib/python3.14/site-packages/zmq/eventloop/zmqstream.py:563]>
Task exception was never retrieved
future: <Task finished name='Task-131' coro=<Kernel.shell_main() done, defined at /Users/sarath/work/doc-parser/venv_pdf/lib/python3.14/site-packages/ipykernel/kernelbase.py:597> exception=KeyError('70e854ee-3d55-422c-98d9-479a5e5bc9e6')>
Traceback (most recent call last):
  File "/Users/sarath/work/doc-parser/venv_pdf/lib/python3.14/sit

pymupdf_fast:   59043 chars,  2891 lines
pymupdf4llm:    44743 chars,   376 lines


In [13]:
md_fast

"## Page 1\n\nHubSpot Reports Strong Q4 and Full Year 2025 Results\n\nFebruary 11, 2026\n\nQ4'25 revenue grew 20% on an as-reported basis and 18% in constant currency compared to Q4'24\nFull year 2025 revenue grew 19% on an as-reported basis and 18% in constant currency compared to 2024\n\nCAMBRIDGE, Mass.--(BUSINESS WIRE)--Feb. 11, 2026-- HubSpot, Inc. (NYSE: HUBS), the customer platform for scaling companies, today\nannounced financial results for the fourth quarter and full year ended December 31, 2025.\n\nFinancial Highlights:\n\nRevenue\nFourth Quarter 2025:\n\nTotal revenue was $846.7 million, up 20% on an as-reported basis and 18% in constant\ncurrency compared to Q4'24.\n\nSubscription revenue was $829.0 million, up 21% on an as-reported basis compared to\nQ4'24.\nProfessional services and other revenue was $17.8 million, up 12% on an as-reported\nbasis compared to Q4'24.\n\nFull Year 2025:\n\nTotal revenue was $3.13 billion, up 19% on an as-reported basis and 18% in constant c

In [12]:
md_4llm

"**==> picture [75 x 21] intentionally omitted <==**\n\n## **HubSpot Reports Strong Q4 and Full Year 2025 Results** \n\nFebruary 11, 2026 \n\n_Q4'25 revenue grew 20% on an as-reported basis and 18% in constant currency compared to Q4'24 Full year 2025 revenue grew 19% on an as-reported basis and 18% in constant currency compared to 2024_ \n\nCAMBRIDGE, Mass.--(BUSINESS WIRE)--Feb. 11, 2026-- HubSpot, Inc. (NYSE: HUBS), the customer platform for scaling companies, today announced financial results for the fourth quarter and full year ended December 31, 2025. \n\n## **Financial Highlights:** \n\n## **Revenue** \n\n_Fourth Quarter 2025:_ \n\n- Total revenue was $846.7 million, up 20% on an as-reported basis and 18% in constant currency compared to Q4'24. \n\n   - Subscription revenue was $829.0 million, up 21% on an as-reported basis compared to Q4'24. \n\n   - Professional services and other revenue was $17.8 million, up 12% on an as-reported basis compared to Q4'24. \n\n_Full Year 2025:

In [11]:
# RecursiveChunker with markdown recipe — splits on headings, paragraphs, sentences
chunker = RecursiveChunker.from_recipe("markdown", lang="en")

# ---- EDIT THIS ----
USE_4LLM = False  # False = pymupdf_fast, True = pymupdf4llm
# -------------------

md = md_4llm if USE_4LLM else md_fast
label = 'pymupdf4llm' if USE_4LLM else 'pymupdf_fast'

chunks = chunker.chunk(md)

print(f'{label}: {len(chunks)} chunks')
print(f'Avg tokens: {sum(c.token_count for c in chunks) / len(chunks):.0f}')
print(f'Min tokens: {min(c.token_count for c in chunks)}')
print(f'Max tokens: {max(c.token_count for c in chunks)}')
print()

for i, chunk in enumerate(chunks):
    preview = chunk.text.replace('\n', ' ')
    print("\n")
    print(f'  Chunk {i:>3} [{chunk.token_count:>5} tok]: {preview}...')

pymupdf_fast: 39 chunks
Avg tokens: 1514
Min tokens: 423
Max tokens: 2044



  Chunk   0 [ 1902 tok]: ## Page 1  HubSpot Reports Strong Q4 and Full Year 2025 Results  February 11, 2026  Q4'25 revenue grew 20% on an as-reported basis and 18% in constant currency compared to Q4'24 Full year 2025 revenue grew 19% on an as-reported basis and 18% in constant currency compared to 2024  CAMBRIDGE, Mass.--(BUSINESS WIRE)--Feb. 11, 2026-- HubSpot, Inc. (NYSE: HUBS), the customer platform for scaling companies, today announced financial results for the fourth quarter and full year ended December 31, 2025.  Financial Highlights:  Revenue Fourth Quarter 2025:  Total revenue was $846.7 million, up 20% on an as-reported basis and 18% in constant currency compared to Q4'24.  Subscription revenue was $829.0 million, up 21% on an as-reported basis compared to Q4'24. Professional services and other revenue was $17.8 million, up 12% on an as-reported basis compared to Q4'24.  Full Year 2025:  Total reven

In [6]:
# Also try with custom chunk_size for finer control
chunker_512 = RecursiveChunker(
    tokenizer='o200k_base',
    chunk_size=512,
    min_characters_per_chunk=24,
)

chunks_512 = chunker_512.chunk(md)

print(f'{label} (chunk_size=512): {len(chunks_512)} chunks')
print(f'Avg tokens: {sum(c.token_count for c in chunks_512) / len(chunks_512):.0f}')
print(f'Min tokens: {min(c.token_count for c in chunks_512)}')
print(f'Max tokens: {max(c.token_count for c in chunks_512)}')
print()

# Check if key financial values land in chunks
keywords = ['846.7', '3,854,151', '288,706', '709,045', '884,945', '1.0 billion']
print('Keyword coverage:')
for kw in keywords:
    found_in = [i for i, c in enumerate(chunks_512) if kw in c.text]
    print(f'  {kw:<15} -> chunk(s) {found_in if found_in else "NOT FOUND"}')

pymupdf4llm (chunk_size=512): 35 chunks
Avg tokens: 369
Min tokens: 34
Max tokens: 511

Keyword coverage:
  846.7           -> chunk(s) [0]
  3,854,151       -> chunk(s) [10]
  288,706         -> chunk(s) [2]
  709,045         -> chunk(s) [12]
  884,945         -> chunk(s) [16, 17, 18]
  1.0 billion     -> chunk(s) [3]


## Interactive: Retrieve top chunks + LLM answer for a question

In [15]:
import numpy as np
from experiments.chunking.embedder import Embedder, cosine_similarity, BM25Index, hybrid_scores
from experiments.chunking.eval_chunking import llm_eval_answer
from parsers.chunker import chunk_markdown, Chunk

PARSERS_ROOT = Path('/Users/sarath/work/doc-parser/parsers')

In [16]:
# ---- EDIT THIS ----
DOC_NAME = 'meta_10q'
# -------------------

md_path = PARSERS_ROOT / 'output' / 'pipeline' / DOC_NAME / f'{DOC_NAME}.md'
chunks_path = PARSERS_ROOT / 'output' / 'pipeline' / DOC_NAME / f'{DOC_NAME}_chunks.json'

# Load pre-computed chunks if available, otherwise chunk from markdown
if chunks_path.exists():
    chunks_data = json.load(open(chunks_path))
    pipeline_chunks = [
        Chunk(text=c['text'], chunk_type=c['chunk_type'],
              page_num=c['page_num'], token_count=c['token_count'])
        for c in chunks_data['chunks']
    ]
    print(f'Loaded {len(pipeline_chunks)} pre-computed chunks from {chunks_path.name}')
else:
    md_pipeline = md_path.read_text()
    pipeline_chunks = chunk_markdown(md_pipeline)
    print(f'Chunked {md_path.name} -> {len(pipeline_chunks)} chunks')

chunk_texts = [c.text for c in pipeline_chunks]
embedder = Embedder()
chunk_embeddings = embedder.embed_texts(chunk_texts)
bm25 = BM25Index(chunk_texts)

print(f'{len(pipeline_chunks)} chunks, embeddings ready')

Loaded 298 pre-computed chunks from meta_10q_chunks.json
298 chunks, embeddings ready


In [17]:
# ---- EDIT THIS ----
QUESTION = "What was HubSpot's total assets as of December 31, 2025?"
EXPECTED = ['3,854,151']

QUESTION = "What is the Beta Adj Exposure for SPX Index at 1 year?"
EXPECTED = ['512.3M']

QUESTION = "What was the fund's Sharpe ratio period to date?"
EXPECTED = ['-7.28']

QUESTION = "what was the income from operations for 3 months ended september 30, 2025"
EXPECTED = ['20,535']

TOP_K = 5
# -------------------

# Retrieve
q_emb = embedder.embed_query(QUESTION)
dense = cosine_similarity(q_emb, chunk_embeddings)
bm25_scores = bm25.score(QUESTION)
scores = hybrid_scores(dense, bm25_scores)
top_idx = np.argsort(scores)[::-1][:TOP_K]

# Show retrieved chunks
print(f'Question: {QUESTION}')
print(f'Expected: {EXPECTED}')
print(f'{"="*80}\n')

for rank, idx in enumerate(top_idx):
    c = pipeline_chunks[idx]
    has_kw = any(kw.lower() in c.text.lower() for kw in EXPECTED)
    marker = ' << HAS KEYWORD' if has_kw else ''
    preview = c.text[:300].replace('\n', ' ')
    print(f'  #{rank+1} [page {c.page_num}, {c.chunk_type}, {c.token_count} tok]{marker}')
    print(f'  {preview}...\n')

# LLM answer
top_texts = [pipeline_chunks[i].text for i in top_idx]
hit, answer = llm_eval_answer(QUESTION, top_texts, EXPECTED)

print(f'{"="*80}')
print(f'LLM Answer (hit={hit}):')
print(answer)

Question: what was the income from operations for 3 months ended september 30, 2025
Expected: ['20,535']

  #1 [page 48, text, 143 tok]
  ### Cash Provided by Operating Activities  Cash provided by operating activities during the nine months ended September 30, 2025 mostly consisted of $37.69 billion net income adjusted for certain non-cash items, such as $17.70 billion of deferred income taxes, $14.54 billion of share-based compensat...

  #2 [page 46, text, 157 tok]
  ## Family of Apps  FoA income from operations in the three and nine months ended September 30, 2025 increased $3.19 billion, or 15%, and $12.92 billion, or 22%, respectively, compared to the same periods in 2024. The increases in FoA income from operations were driven by higher advertising revenue w...

  #3 [page 43, table, 363 tok] << HAS KEYWORD
  The following table sets forth our condensed consolidated statements of income data (in millions): | | Three Months Ended September 30, | | Nine Months Ended September 30, 